# Full-Text Search Patterns

## Overview
This notebook applies FTS to real clinical search scenarios: patient chart search, literature retrieval, autocomplete, multilingual content (English + French for Canadian healthcare), and faceted search combining FTS with SQL aggregation.

**Pattern summary:**

| Pattern | Use case |
|---|---|
| Patient chart search | Find all notes for a patient mentioning a term |
| Literature search | Rank research articles by relevance with snippets |
| Autocomplete | Prefix queries as user types (`diab:*`) |
| Multilingual | English + French tsvector columns |
| Faceted search | FTS filter + GROUP BY for result category counts |
| Combined config | `english` for narrative + `simple` for acronyms/drug names |

---

In [ ]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Clinical notes and patient records for FTS demos
CREATE TABLE patients (
    patient_id  TEXT PRIMARY KEY,
    full_name   TEXT NOT NULL,
    province    TEXT NOT NULL,
    dob         TEXT NOT NULL
);

CREATE TABLE clinical_notes (
    note_id     TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL REFERENCES patients(patient_id),
    note_date   TEXT NOT NULL,
    provider    TEXT NOT NULL,
    note_type   TEXT NOT NULL,  -- 'progress','discharge','consult','operative'
    content     TEXT NOT NULL
);

CREATE TABLE medications (
    med_id      TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL,
    drug_name   TEXT NOT NULL,
    indication  TEXT,
    notes       TEXT
);

CREATE TABLE research_articles (
    article_id  TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    abstract    TEXT NOT NULL,
    keywords    TEXT,
    published   TEXT NOT NULL
);

-- FTS5 virtual tables (SQLite equivalent of PostgreSQL tsvector index)
CREATE VIRTUAL TABLE notes_fts USING fts5(
    note_id UNINDEXED,
    patient_id UNINDEXED,
    note_type UNINDEXED,
    content,
    tokenize='porter ascii'   -- porter stemmer (closest to pg's english dictionary)
);

CREATE VIRTUAL TABLE articles_fts USING fts5(
    article_id UNINDEXED,
    title,
    abstract,
    keywords,
    tokenize='porter ascii'
);
""")

patients = [
    ('P001','Alice Ngata',      'NB','1980-03-15'),
    ('P002','Bob Chen',         'ON','1972-07-22'),
    ('P003','Fatima Rashid',    'BC','1986-11-05'),
    ('P004','James MacLeod',    'NS','1963-01-30'),
    ('P005','Mei-Ling Tran',    'QC','1966-08-19'),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?)", patients)

notes = [
    ('N001','P001','2023-10-01','Dr. Lee','progress',
     'Patient presents with poorly controlled hypertension. Blood pressure 148/92. '
     'Currently on Lisinopril 10mg once daily. Discussed medication adherence and sodium restriction. '
     'Patient reports occasional dizziness and dry cough, a known side effect of ACE inhibitors. '
     'Recommended dietary changes and increased physical activity. Follow-up in 4 weeks.'),
    ('N002','P001','2024-01-15','Dr. Lee','progress',
     'Follow-up for hypertension management. Blood pressure improved to 132/84. '
     'Patient adherent to Lisinopril. Dry cough persists. Considering switching to ARB if cough continues. '
     'HbA1c 7.2%, borderline elevated. Discussed diabetes prevention strategies. '
     'Lipid panel ordered. Continue current antihypertensive therapy.'),
    ('N003','P002','2024-01-10','Dr. Pham','discharge',
     'Patient admitted for acute exacerbation of Type 2 Diabetes. HbA1c 8.4% on admission. '
     'Adjusted Metformin to 1000mg BID and added Empagliflozin 10mg daily. '
     'Nutritional counselling provided. Patient educated on carbohydrate counting and glucose monitoring. '
     'Discharged in stable condition with follow-up appointment in 2 weeks.'),
    ('N004','P003','2023-08-20','Dr. Singh','consult',
     'Respiratory consultation for persistent asthma exacerbations. '
     'Patient reports frequent nocturnal wheezing and dyspnoea on exertion. '
     'Peak flow measurements show significant variability. Spirometry confirms obstructive pattern. '
     'Current inhaler technique reviewed and corrected. Prescribed Fluticasone-Salmeterol combination inhaler. '
     'Referral for pulmonary rehabilitation. Advised to avoid known allergens including dust mites and pet dander.'),
    ('N005','P004','2023-09-01','Dr. Lee','progress',
     'Routine diabetic review. HbA1c 7.8%, improved from last visit. '
     'eGFR 72 mL/min, stable kidney function. Patient reports no symptoms of hypoglycaemia. '
     'Foot examination normal. Retinal screening up to date. '
     'Continue current diabetes management. Annual flu vaccination administered. Blood pressure well controlled.'),
    ('N006','P005','2024-02-01','Dr. Pham','progress',
     'Complex case: Type 2 Diabetes with Hypertension and CKD Stage 3. '
     'HbA1c 9.1%, suboptimal glycaemic control. eGFR 38 mL/min, declining renal function. '
     'Metformin contraindicated due to renal impairment. Switched to Insulin glargine 20 units nocte. '
     'Potassium 5.8 mmol/L, hyperkalaemia noted. Dietary potassium restriction advised. '
     'Nephrology referral placed. Strict blood pressure control essential to slow CKD progression.'),
    ('N007','P002','2023-05-15','Dr. Pham','operative',
     'Pre-operative assessment for elective cholecystectomy. '
     'Medical history: Type 2 Diabetes, Hypertension, well controlled on Metformin and Lisinopril. '
     'Electrocardiogram normal. Chest X-ray clear. Blood glucose within acceptable range. '
     'Patient advised to withhold Metformin 48 hours prior to surgery due to contrast risk. '
     'Anaesthesia consultation arranged. Patient consented and cleared for surgery.'),
    ('N008','P001','2024-03-15','Dr. Singh','consult',
     'Cardiology consult for management of resistant hypertension. '
     'Despite optimal doses of Lisinopril and Amlodipine, blood pressure remains elevated at 158/96. '
     'Echocardiogram shows mild left ventricular hypertrophy. '
     'Added Spironolactone 25mg daily for additional blood pressure reduction. '
     'Stress test recommended to rule out ischaemic heart disease. '
     'Patient to monitor blood pressure at home twice daily and maintain log.'),
]
conn.executemany("INSERT INTO clinical_notes VALUES (?,?,?,?,?,?)", notes)

articles = [
    ('A001','Management of Type 2 Diabetes with Renal Impairment',
     'Type 2 diabetes mellitus complicated by chronic kidney disease requires careful medication selection. '
     'Metformin should be avoided when eGFR falls below 30 mL/min due to risk of lactic acidosis. '
     'SGLT2 inhibitors demonstrate renoprotective effects and cardiovascular benefits. '
     'Insulin therapy remains an option at all stages of renal impairment.',
     'diabetes,CKD,renal impairment,Metformin,SGLT2 inhibitor','2023-06-01'),
    ('A002','Hypertension Treatment Guidelines: ACE Inhibitors and ARBs',
     'Angiotensin-converting enzyme inhibitors and angiotensin receptor blockers are first-line '
     'antihypertensive agents. ACE inhibitors commonly cause dry cough due to bradykinin accumulation. '
     'ARBs are preferred in patients who cannot tolerate ACE inhibitor cough. '
     'Both drug classes provide renoprotection in diabetic nephropathy.',
     'hypertension,ACE inhibitor,ARB,Lisinopril,blood pressure','2022-11-15'),
    ('A003','Asthma Exacerbation Management in Primary Care',
     'Acute asthma exacerbations require rapid assessment of severity using peak expiratory flow and '
     'oxygen saturation. Short-acting beta-agonists remain the cornerstone of acute bronchodilation. '
     'Inhaled corticosteroids reduce airway inflammation and prevent future exacerbations. '
     'Patients with frequent exacerbations should be referred for specialist pulmonary assessment.',
     'asthma,exacerbation,bronchodilator,corticosteroid,peak flow','2023-03-20'),
    ('A004','Glycaemic Control Targets in Hospitalised Patients',
     'Inpatient hyperglycaemia is associated with increased morbidity, infection risk, and length of stay. '
     'Target blood glucose range of 6-10 mmol/L is recommended for most hospitalised patients. '
     'Insulin infusion protocols allow precise glycaemic management in critically ill patients. '
     'HbA1c measurement on admission provides information about pre-admission glycaemic control.',
     'glycaemic control,HbA1c,insulin,hospitalisation,hyperglycaemia','2024-01-08'),
    ('A005','Chronic Kidney Disease Progression and Blood Pressure Control',
     'Hypertension is both a cause and consequence of chronic kidney disease. '
     'Strict blood pressure control below 130/80 mmHg slows CKD progression significantly. '
     'Renin-angiotensin-aldosterone system blockade with ACE inhibitors or ARBs is recommended '
     'for patients with proteinuria. Regular monitoring of eGFR and potassium is essential.',
     'CKD,hypertension,blood pressure,ACE inhibitor,eGFR,proteinuria','2023-09-12'),
]
conn.executemany("INSERT INTO research_articles VALUES (?,?,?,?,?)", articles)

meds = [
    ('M001','P001','Lisinopril',   'Hypertension','10mg once daily'),
    ('M002','P001','Amlodipine',   'Hypertension','5mg once daily'),
    ('M003','P002','Metformin',    'Type 2 Diabetes','500mg BID'),
    ('M004','P002','Lisinopril',   'Hypertension','5mg once daily'),
    ('M005','P003','Salbutamol',   'Asthma','PRN inhaler'),
    ('M006','P005','Insulin glargine','Type 2 Diabetes','20 units nocte'),
    ('M007','P004','Metformin',    'Type 2 Diabetes','500mg daily'),
]
conn.executemany("INSERT INTO medications VALUES (?,?,?,?,?)", meds)

# Populate FTS5 indexes
conn.execute("INSERT INTO notes_fts SELECT note_id,patient_id,note_type,content FROM clinical_notes")
conn.execute("""
    INSERT INTO articles_fts
    SELECT article_id, title, abstract, COALESCE(keywords,'') FROM research_articles
""")
conn.commit()

print("FTS healthcare dataset loaded:")
for tbl in ['patients','clinical_notes','medications','research_articles']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<22s}: {n} rows")
print("  notes_fts (FTS5)      : indexed")
print("  articles_fts (FTS5)   : indexed")


---
## Autocomplete and typeahead

In [ ]:
print("=== Autocomplete / typeahead with FTS ===")
print()

# Prefix search simulating autocomplete
print("Autocomplete: prefix search as user types")
partial_inputs = ['dia', 'hyper', 'ren', 'HbA', 'inhal']
print(f"  {'partial input':<16s}  {'notes':>6s}  {'articles':>9s}")
print("  " + "-"*36)
for partial in partial_inputs:
    query = partial + '*'
    notes_n = conn.execute(
        "SELECT COUNT(*) FROM notes_fts WHERE notes_fts MATCH ?", (query,)
    ).fetchone()[0]
    arts_n = conn.execute(
        "SELECT COUNT(*) FROM articles_fts WHERE articles_fts MATCH ?", (query,)
    ).fetchone()[0]
    print(f"  '{partial}'  ({query:<12s})  {notes_n:>6d}  {arts_n:>9d}")

print()
print("PostgreSQL autocomplete with prefix tsquery:")
print("""
  -- As user types 'diab', search with prefix:
  SELECT note_id,
         ts_headline('english', content,
                     to_tsquery('english', 'diab:*'),
                     'MaxWords=10, MinWords=5') AS preview
  FROM   clinical_notes
  WHERE  tsv @@ to_tsquery('english', 'diab:*')
  ORDER  BY ts_rank(tsv, to_tsquery('english', 'diab:*')) DESC
  LIMIT  5;

  -- Autocomplete suggestions from a terms table:
  CREATE TABLE search_terms AS
  SELECT DISTINCT unnest(tsvector_to_array(tsv)) AS term
  FROM   clinical_notes;
  CREATE INDEX ON search_terms(term text_pattern_ops);

  -- Then for prefix 'hyper':
  SELECT term FROM search_terms
  WHERE  term LIKE 'hyper%'
  ORDER  BY term
  LIMIT  10;
""")


---
## Multilingual search

In [ ]:
print("=== Multilingual search and language detection ===")
print()

print("FTS with multiple languages (PostgreSQL):")
print("""
  -- Table with explicit language column:
  CREATE TABLE patient_notes_multi (
      note_id   TEXT PRIMARY KEY,
      content   TEXT NOT NULL,
      language  TEXT NOT NULL DEFAULT 'english',  -- 'english','french','simple'
      tsv       tsvector
  );

  -- Trigger that uses per-row language:
  CREATE OR REPLACE FUNCTION update_multilang_tsv() RETURNS trigger AS $$
  BEGIN
      NEW.tsv := to_tsvector(NEW.language::regconfig, NEW.content);
      RETURN NEW;
  END;
  $$ LANGUAGE plpgsql;

  -- Query with dynamic language:
  SELECT note_id FROM patient_notes_multi
  WHERE  to_tsvector(language::regconfig, content)
         @@ plainto_tsquery(language::regconfig, 'diabète');
  -- Works for French notes using French stemming
""")

print()
print("Handling bilingual content (English + French for Canadian healthcare):")
print("""
  -- Combine both language vectors:
  ALTER TABLE clinical_notes
      ADD COLUMN tsv_en tsvector  -- English stemmed
          GENERATED ALWAYS AS (to_tsvector('english', content)) STORED,
      ADD COLUMN tsv_fr tsvector  -- French stemmed
          GENERATED ALWAYS AS (to_tsvector('french', content)) STORED;

  CREATE INDEX ON clinical_notes USING GIN (tsv_en);
  CREATE INDEX ON clinical_notes USING GIN (tsv_fr);

  -- Search both:
  SELECT note_id FROM clinical_notes
  WHERE  tsv_en @@ to_tsquery('english', 'diabetes')
      OR tsv_fr @@ to_tsquery('french',  'diabète');
""")

print()
print("'simple' config for acronyms and drug names:")
print("""
  -- 'english' lowercases everything: ACE → 'ace', HbA1c → 'hba1c'
  -- 'simple' preserves exact tokens (no stemming, no stopwords)

  -- For clinical search, a combined config:
  CREATE INDEX idx_notes_simple ON clinical_notes
      USING GIN (to_tsvector('simple', content));

  -- Prefix search for drug names (case-insensitive, no stemming):
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('simple', LOWER(content))
         @@ to_tsquery('simple', 'lisinopril:*');
""")


---
## Real-world clinical search patterns

In [ ]:
print("=== Real-world FTS patterns: clinical search ===")
print()

# Pattern 1: Search across patients + notes joined
rows = conn.execute("""
    SELECT p.patient_id, p.full_name, n.note_id, n.note_date, n.note_type,
           snippet(notes_fts, 3, '>>','<<', '...', 15) AS snip
    FROM   notes_fts
    JOIN   clinical_notes n ON n.note_id = notes_fts.note_id
    JOIN   patients p ON p.patient_id = n.patient_id
    WHERE  notes_fts MATCH 'HbA1c'
    ORDER  BY notes_fts.rank
""").fetchall()
print("Pattern 1 — Patient chart search for 'HbA1c':")
print(f"  {'patient':<12s}  {'note_id':<8s}  {'date':<12s}  {'type':<10s}  snippet")
print("  " + "-"*75)
for r in rows:
    print(f"  {r['full_name']:<12s}  {r['note_id']:<8s}  {r['note_date']:<12s}  "
          f"{r['note_type']:<10s}  {r['snip'][:45]}")

print()
# Pattern 2: Search articles, return ranked with snippet
rows2 = conn.execute("""
    SELECT a.article_id, a.title, a.published,
           ROUND(-articles_fts.rank, 4) AS score,
           snippet(articles_fts, 1, '[',']','...',20) AS snip
    FROM   articles_fts
    JOIN   research_articles a ON a.article_id = articles_fts.article_id
    WHERE  articles_fts MATCH 'renal OR kidney OR CKD'
    ORDER  BY articles_fts.rank
""").fetchall()
print("Pattern 2 — Literature search for 'renal OR kidney OR CKD':")
for r in rows2:
    print(f"  {r['article_id']}  score={r['score']:.4f}  [{r['published']}]")
    print(f"     Title: {r['title']}")
    print(f"     Snip:  {r['snip'][:80]}")
    print()

print("Pattern 3 — PostgreSQL: full patient search with ranking and highlighting:")
print("""
  WITH query AS (
      SELECT websearch_to_tsquery('english', 'diabetes renal impairment') AS q
  )
  SELECT p.patient_id,
         p.full_name,
         n.note_date,
         n.note_type,
         ts_rank_cd(n.tsv, q.q, 32)  AS relevance,
         ts_headline('english', n.content, q.q,
             'MaxWords=25, MinWords=10, MaxFragments=2,
              StartSel=<mark>, StopSel=</mark>') AS excerpt
  FROM   clinical_notes n
  JOIN   patients p ON p.patient_id = n.patient_id,
         query q
  WHERE  n.tsv @@ q.q
  ORDER  BY relevance DESC
  LIMIT  10;
""")

print("Pattern 4 — Faceted search with FTS + SQL aggregation:")
print("""
  WITH matches AS (
      SELECT n.note_id, n.note_type, n.patient_id
      FROM   clinical_notes n
      WHERE  n.tsv @@ websearch_to_tsquery('english', 'hypertension')
  )
  -- Facet by note type:
  SELECT note_type,
         COUNT(*) AS match_count
  FROM   matches
  GROUP  BY note_type
  ORDER  BY match_count DESC;
""")


---
## Common Pitfalls

**1. Autocomplete prefix search scanning the entire table**
A prefix query `to_tsquery('english', 'diab:*')` uses the GIN index, but generating autocomplete suggestions by scanning tsvector columns is expensive. For fast autocomplete, maintain a separate `search_terms` table of distinct lexemes with a B-tree index using `text_pattern_ops` for `LIKE 'prefix%'` queries.

**2. Not validating language detection for multilingual content**
Automatic language detection (via external libraries like `langdetect`) can misclassify short clinical texts that mix English and French abbreviations. A misidentified language applies the wrong stemmer, causing missed matches. For Canadian healthcare data, default to English config unless the record is explicitly marked as French.

**3. Faceted search with FTS + GROUP BY not using the FTS index**
A query that does `WHERE tsv @@ query GROUP BY note_type` will use the GIN index for the FTS filter, but the GROUP BY is performed on the full match set. For large match sets, pre-aggregating facet counts via a materialized view (refreshed nightly) is faster than computing them on every search request.

**4. Search returning zero results for valid clinical terms**
The 'english' config stems 'ACE' to 'ace', 'eGFR' to 'egfr', and drops numeric tokens like '7.2'. A clinician searching for 'HbA1c 7.2' gets no results because '7.2' is not indexed. Use a combined index: `to_tsvector('english', content) || to_tsvector('simple', content)` to capture both stemmed narrative and exact clinical tokens.

**5. ts_headline called for every row in a large result set**
`ts_headline()` is computationally expensive — it re-parses the full document text to find and highlight fragments. Calling it for 10,000 search results is very slow. Apply `LIMIT` before calling `ts_headline`, or generate headlines only for the top page of results (e.g., `LIMIT 10`).

**6. Building a patient search that returns records from other patients**
Full-text search across `clinical_notes` without a `WHERE patient_id = $1` filter returns all patients' matching notes. In healthcare applications, FTS must always be scoped to the authorised patient or access list. Never expose cross-patient search to end users without explicit access control checks.


---
*sql_methods_library - Samantha McGarrigle*